In [1]:
from dotenv import load_dotenv
import os
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

In [3]:
load_dotenv()

True

In [ ]:
!pip install -qU "langchain-chroma>0.1.2"

In [4]:
from langchain_community.vectorstores import Chroma


In [ ]:
#chain = prompt | llm

In [5]:
import langchain

In [ ]:
!pip install langchainhub

In [6]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic import hub 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
pip show langchain


In [ ]:
messages = [("system","You are a helpful assistant that gives responses in queries"),
("user","Hello, my name is ebuka, what kind of queries can you help with?")]

In [ ]:
prompt = ChatPromptTemplate.from_messages(messages)

In [ ]:
prompt

In [ ]:
llm = ChatGroq?

In [7]:
llm = ChatGroq(model = "meta-llama/llama-4-scout-17b-16e-instruct",
temperature = 0.3,
api_key = os.environ.get("GROQ_API_KEY"))

In [ ]:
chain = prompt | llm | StrOutputParser

In [ ]:
print(chain.invoke)

In [ ]:
messages = [("system","You are a helpful assistant that gives responses in queries"),
MessagesPlaceholder("history")]

prompt = ChatPromptTemplate.from_messages(messages)
chain = prompt | llm | StrOutputParser()

In [ ]:
history = []
while True:
    user_input = input("You: ")
    history.append(("user",user_input))
    response = chain.invoke({"input": user_input,"history":history})
    print("Bot: ",response)
    history.append(("assistant", response))

    if user_input == "quit":
        break


# Building a RAG

In [8]:
loader = TextLoader(r"C:\Users\chukw\Documents\RAIN\AIML 2ND SEMESTER (File responses)\Rain Class\Week 10 LLM\about rain.txt")
documents = loader.load()
documents 

[Document(metadata={'source': 'C:\\Users\\chukw\\Documents\\RAIN\\AIML 2ND SEMESTER (File responses)\\Rain Class\\Week 10 LLM\\about rain.txt'}, page_content="CHECK FOR PART-SCHOLARSHIP OPPORTUNITY\n\nROBOTICS & ARTIFICIAL INTELLIGENCE NIGERIA\nHome\nBECOME A TRAINEE\nCORPORATE SERVICES\nMore\n+2348114276861 info@raiNIGERIA.com\n\nRobotics & Artificial Intelligence Nigeria\n\nWORLD CLASS TRAINING\nAdmission into MAY/JUNE 2026 - Cohort 20 has Commenced. Apply NOW. Click here\n\nCLICK TO APPLY\n\nWhy RAIN ?\n\n12-Months Intensive Courses in Machine Learning and Robotics\nLearning from some of the best faculties, and using the most up-to-date curriculum in AI and Robotics, RAIN provides a 12-Months 3-Semester in-Class intensive course in Robot Development and Automation, as well as in Artificial Intelligence and Machine Learning.\n\nAPPLY\n\n\nLearn Product Development from the Experts\nAt this point in your life, you don't want boring classroom sessions. The universities are there for th

In [9]:
#USing Recursive text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,chunk_overlap = 100)
chunks = text_splitter.split_documents(documents)
chunks

[Document(metadata={'source': 'C:\\Users\\chukw\\Documents\\RAIN\\AIML 2ND SEMESTER (File responses)\\Rain Class\\Week 10 LLM\\about rain.txt'}, page_content='CHECK FOR PART-SCHOLARSHIP OPPORTUNITY\n\nROBOTICS & ARTIFICIAL INTELLIGENCE NIGERIA\nHome\nBECOME A TRAINEE\nCORPORATE SERVICES\nMore\n+2348114276861 info@raiNIGERIA.com\n\nRobotics & Artificial Intelligence Nigeria\n\nWORLD CLASS TRAINING\nAdmission into MAY/JUNE 2026 - Cohort 20 has Commenced. Apply NOW. Click here\n\nCLICK TO APPLY\n\nWhy RAIN ?\n\n12-Months Intensive Courses in Machine Learning and Robotics\nLearning from some of the best faculties, and using the most up-to-date curriculum in AI and Robotics, RAIN provides a 12-Months 3-Semester in-Class intensive course in Robot Development and Automation, as well as in Artificial Intelligence and Machine Learning.\n\nAPPLY'),
 Document(metadata={'source': 'C:\\Users\\chukw\\Documents\\RAIN\\AIML 2ND SEMESTER (File responses)\\Rain Class\\Week 10 LLM\\about rain.txt'}, page

In [ ]:
len(chunks)

In [ ]:
chunks[0]

In [ ]:
chunks[2]

In [ ]:
#!pip install langchain-pinecone



In [ ]:
#import pinecone

#pinecone.init(
    
    #api_key = os.environ.get("GROQ_API_KEY")# e.g., "us-west1-gcp"
#)

In [ ]:
#import pinecone
#from langchain_pinecone import PineconeVectorStore
#from langchain_openai import OpenAIEmbeddings

In [10]:
from langchain_community.vectorstores import Chroma

from langchain_openai import OpenAIEmbeddings

In [11]:
from langchain_chroma import Chroma

In [12]:
#Initialize Embeddings
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small",dimensions = 512)
#Initialize Chroma Vector Store
vector_store = Chroma.from_documents(chunks,embeddings, persist_directory = "chromarag.db")


In [13]:
retriever = vector_store.as_retriever()

In [14]:
history = []
rewrite_question_prompt  = ChatPromptTemplate.from_messages(
    [("system","""
    you are a helpful assistant, based on this conversation history, rewrite the user's question as a stand alone question.
    {chat_history}
    """),
    MessagesPlaceholder("chat_history"),
    ("user","{input}")]
)

In [15]:
llm = ChatGroq(temperature=0.3, model="meta-llama/llama-4-scout-17b-16e-instruct")

In [16]:
history_aware_retriever = create_history_aware_retriever(llm,retriever,rewrite_question_prompt)

In [17]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", """
        You are a helpful assistant. Given the context, answer the user's question. IF you do not know the answer state that you do not know the answer. Never hallucinate.
        {context}
        """),
        MessagesPlaceholder("chat_history"),
        ("user",'{input}')
    ]
)

In [18]:
stuff_chain = create_stuff_documents_chain(llm,qa_prompt)

In [19]:
chain = create_retrieval_chain(history_aware_retriever,stuff_chain)

In [20]:
response = chain.invoke({"input": "What does RAIN do?", "chat_history":history})

In [21]:
response['answer']

'RAIN (Robotics & Artificial Intelligence Nigeria) is a research and training facility dedicated to robotics and artificial intelligence technologies. Specifically, RAIN handles training and certifications in areas such as:\n\n* Machine Learning (ML)\n* Deep Learning for Computer Vision\n* Natural Language Processing\n\nRAIN provides training programs with a maximum of 30 trainees per cohort, and offers certifications upon completion of these programs. The organization is based in Ibadan, Nigeria, and operates with a team of highly skilled developers and academics.'

In [22]:
response = chain.invoke({"input": "What does FIRS do?", "chat_history":history})

In [ ]:
response['answer']

### CONVERSATION

In [23]:
while True:
    user_input = input("you: ")
    history.append(("user", user_input))
    response = chain.invoke({"input":user_input, "chat_history":history})
    print(response["answer"])
    history.append(("assistant",response["answer"]))

    if user_input == "quit":
        break


RAIN is a research and training facility dedicated to robotics and artificial intelligence (AI) technologies. Here are some key facts about RAIN:

1. **Location**: RAIN operates out of its research and training facility located in Ibadan, Nigeria.
2. **Mission**: RAIN is focused on providing high-quality training and certification in robotics and AI technologies.
3. **Training programs**: RAIN offers training cohort programs with an average of 30 trainees per cohort. The programs are led by a host of highly skilled developers and academics.
4. **Global recognition**: RAIN has gained recognition on a global scale and across different standards, presenting high competency in the field of robotics and AI.
5. **Notable visitors**: RAIN has welcomed foreign diplomats to its facility, including the Deputy Secretary General of the United Nations, the Canadian Trade Commissioner to Nigeria, and delegations from the United States Consulate and the United Nations Development Programme (UNDP) Nig